Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise 2.3: Schema and Partition Evolution

How Iceberg handles table evolution on the NYC Taxi dataset:
- **Schema evolution** — add/remove/rename columns without rewriting data
- **Field IDs** — internal IDs that prevent *data resurrection* (deleted column data reappearing when a name is reused)
- **Partition evolution** — change partitioning strategy over time

⚠️ **Spark Connect**: one Spark server in the background, notebooks are thin clients. If `ConnectionRefusedError` appears, check `docker logs jupyter-spark` or restart with `docker compose restart jupyter`.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("SchemaAndPartitionEvolution") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.evolution")
print("Namespace 'evolution' created!")

## Helper Functions: PyIceberg Table API

Spark SQL doesn't expose Iceberg internals like field IDs or partition-spec history. We use **[PyIceberg](https://py.iceberg.apache.org/)** to talk to the Polaris REST catalog directly over HTTP — same API any engine can use — for lower-level metadata (schemas with field IDs, full partition-spec history, column defaults).

> Catalog credentials (`POLARIS_CLIENT_ID` / `POLARIS_CLIENT_SECRET`) are set by the Docker environment.

In [ ]:
import os
from pyiceberg.catalog import load_catalog

catalog = load_catalog("polaris", **{
    "type": "rest",
    "uri": "http://polaris:8181/api/catalog",
    "credential": f"{os.environ['POLARIS_CLIENT_ID']}:{os.environ['POLARIS_CLIENT_SECRET']}",
    "scope": "PRINCIPAL_ROLE:ALL",
    "warehouse": "quickstart_catalog",
})

def load_iceberg_table(table_name):
    """Load an Iceberg Table object via the PyIceberg REST catalog."""
    parts = table_name.replace("polaris.", "").split(".")
    return catalog.load_table(f"{parts[0]}.{parts[1]}")

def show_iceberg_schema(table_name, highlight_field_id=None, highlight_label=None):
    """Print the Iceberg schema with field IDs.
    
    Optionally highlight a specific field ID with a label.
    If the highlighted field ID is not present, a note is printed at the end.
    """
    table = load_iceberg_table(table_name)
    found_highlight = False
    print(f"{'ID':>4}  {'Name':<30} {'Type':<12} {'Optional'}")
    print("-" * 60)
    for field in table.schema().fields:
        marker = ""
        if highlight_field_id is not None and field.field_id == highlight_field_id:
            found_highlight = True
            marker = f"  <-- {highlight_label or 'highlighted'}"
        print(f"{field.field_id:>4}  {field.name:<30} {str(field.field_type):<12} {field.optional}{marker}")
    if highlight_field_id is not None and not found_highlight:
        print(f"\n  * Field ID {highlight_field_id} is absent: {highlight_label or 'not in schema'}")

def show_partition_specs(table_name, highlight_spec_id=None, highlight_label=None):
    """Print all partition specs for an Iceberg table, marking the current one."""
    table = load_iceberg_table(table_name)
    specs = table.specs()
    current_id = table.spec().spec_id
    print(f"{'Spec ID':<10} {'Fields':<50} {'Status'}")
    print("-" * 70)
    for spec_id in sorted(specs):
        spec = specs[spec_id]
        fields = []
        for f in spec.fields:
            fields.append(f"{f.transform}({table.schema().find_column_name(f.source_id)})")
        field_str = ", ".join(fields) if fields else "(unpartitioned)"
        tags = []
        if spec_id == current_id:
            tags.append("current")
        if highlight_spec_id is not None and spec_id == highlight_spec_id:
            tags.append(highlight_label or "highlighted")
        status = f"  <-- {', '.join(tags)}" if tags else ""
        print(f"{spec_id:<10} {field_str:<50} {status}")

def add_column_with_default(table_name, col_name, iceberg_type, default_value):
    """Add a column with an initial default via PyIceberg.
    
    The initial default is applied retroactively: existing rows that were
    written before the column existed will return this value instead of NULL.
    Requires format-version 3.
    """
    table = load_iceberg_table(table_name)
    with table.update_schema() as update:
        update.add_column(col_name, iceberg_type, default_value=default_value)
    spark.sql(f"REFRESH TABLE {table_name}")

def show_column_defaults(table_name):
    """Print columns that have initial or write defaults."""
    table = load_iceberg_table(table_name)
    has_any = False
    for field in table.schema().fields:
        initial = field.initial_default
        write = field.write_default
        if initial is not None or write is not None:
            has_any = True
            print(f"  {field.name}: initialDefault={initial}, writeDefault={write}")
    if not has_any:
        print("  (no columns have defaults)")

print("PyIceberg helpers loaded!")

## Download NYC Taxi Data

Yellow Taxi trips, **June–September 2023**. May already be in MinIO if you ran earlier exercises.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id=os.environ.get('MINIO_ROOT_USER', 'admin'),
    aws_secret_access_key=os.environ.get('MINIO_ROOT_PASSWORD', 'password'),
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6, 7, 8, 9]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO, skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~45MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nAll taxi data ready in MinIO!")

## Part 1: Schema Evolution — Adding Columns

Schema changes in Iceberg are **metadata-only** — never rewrite data files. Adding/dropping/renaming is instant (unlike traditional `ALTER TABLE` that can force expensive rewrites). The schema updates in metadata; engines apply it at read time.

### Create Table from June Data

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.evolution.nyc_taxi")

spark.sql("""
    CREATE TABLE polaris.evolution.nyc_taxi
    USING iceberg
    TBLPROPERTIES ('format-version' = '3')
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.evolution.nyc_taxi").collect()[0][0]
print(f"Table created with {count:,} trips (format-version 3)")

In [ ]:
print("Initial schema:")
spark.sql("DESCRIBE polaris.evolution.nyc_taxi").show(truncate=False)

### Add a Column - Instant Operation

In [ ]:
start = time.time()

spark.sql("""
    ALTER TABLE polaris.evolution.nyc_taxi
    ADD COLUMN tip_percentage DOUBLE
""")

elapsed = time.time() - start
print(f"Column added in {elapsed:.3f} seconds (metadata-only operation!)")

In [ ]:
print("Schema after adding tip_percentage:")
spark.sql("DESCRIBE polaris.evolution.nyc_taxi").show(truncate=False)

In [ ]:
print("Old rows have NULL for the new column:")
spark.sql("""
    SELECT VendorID, fare_amount, tip_amount, tip_percentage
    FROM polaris.evolution.nyc_taxi
    LIMIT 5
""").show()

### Insert New Data with the New Column Populated

In [ ]:
spark.sql("""
    INSERT INTO polaris.evolution.nyc_taxi
    SELECT *, ROUND(tip_amount / NULLIF(fare_amount, 0) * 100, 2) as tip_percentage
    FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
""")

print("July data inserted with tip_percentage populated!")

In [ ]:
print("New rows have tip_percentage, old rows still have NULL:")
spark.sql("""
    SELECT VendorID, fare_amount, tip_amount, tip_percentage
    FROM polaris.evolution.nyc_taxi
    WHERE tip_percentage IS NOT NULL
    LIMIT 5
""").show()

null_count = spark.sql("SELECT COUNT(*) FROM polaris.evolution.nyc_taxi WHERE tip_percentage IS NULL").collect()[0][0]
filled_count = spark.sql("SELECT COUNT(*) FROM polaris.evolution.nyc_taxi WHERE tip_percentage IS NOT NULL").collect()[0][0]
print(f"Rows with NULL tip_percentage (June): {null_count:,}")
print(f"Rows with tip_percentage (July):      {filled_count:,}")

### Add a Column with an Initial Default

Spark `ADD COLUMN` gives existing rows `NULL`. Iceberg also supports **initial defaults** applied retroactively at read time (no rewrite) — set via the PyIceberg `add_column_with_default` helper over the REST API.

Requires **format-version 3**. V1 = original; V2 = row-level deletes (see E3.1); V3 = initial defaults + default expressions. Most production is on V2; V3 recommended for new tables.

In [ ]:
from pyiceberg.types import DoubleType

add_column_with_default(
    "polaris.evolution.nyc_taxi",
    "surge_multiplier",
    DoubleType(),
    1.0,
)

print("Column 'surge_multiplier' added with initial default = 1.0")
print()
print("Existing rows (written before the column existed) return the default:")
spark.sql("""
    SELECT VendorID, fare_amount, surge_multiplier
    FROM polaris.evolution.nyc_taxi
    LIMIT 5
""").show()

In [ ]:
print("Compare with tip_percentage (added via Spark SQL, no default):")
spark.sql("""
    SELECT VendorID, tip_percentage, surge_multiplier
    FROM polaris.evolution.nyc_taxi
    LIMIT 5
""").show()

print("Column defaults stored in metadata:")
show_column_defaults("polaris.evolution.nyc_taxi")

`tip_percentage` is `NULL` for old rows (no default). `surge_multiplier` shows `1.0` — the initial default is filled at read time, no rewrite.

In [ ]:
spark.sql("ALTER TABLE polaris.evolution.nyc_taxi DROP COLUMN surge_multiplier")
print("Dropped surge_multiplier (cleanup for next section)")

### Try It: Evolve the Schema

Add a column (e.g. `trip_type STRING`, `is_shared_ride BOOLEAN`), insert rows with a value, then query — old rows show `NULL`, new rows have the value. Drop it when done. Bonus: use `add_column_with_default` so old rows get a value too.

In [ ]:
# my_column = "???"
# my_type = "STRING"  # or BOOLEAN, INT, DOUBLE, etc.

# Add the column
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi ADD COLUMN {my_column} {my_type}")
# show_iceberg_schema("polaris.evolution.nyc_taxi")

# Query to see NULL for old rows
# spark.sql(f"SELECT VendorID, fare_amount, {my_column} FROM polaris.evolution.nyc_taxi LIMIT 5").show()

# Clean up when done
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi DROP COLUMN {my_column}")

## Part 2: Field IDs and Data Resurrection Prevention

Iceberg tracks columns by **internal field IDs**, not names. This blocks a subtle bug: drop a column, later re-add one with the same name → old data does **not** come back.

### View Field IDs

In [ ]:
print("Schema with Iceberg field IDs:")
show_iceberg_schema(
    "polaris.evolution.nyc_taxi",
    highlight_field_id=7,
    highlight_label="we will drop this column next"
)

### Drop a Column

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.nyc_taxi
    DROP COLUMN store_and_fwd_flag
""")

print("store_and_fwd_flag column dropped!")

In [ ]:
print("Schema after dropping store_and_fwd_flag (note: field ID 7 is now retired):")
show_iceberg_schema(
    "polaris.evolution.nyc_taxi",
    highlight_field_id=7,
    highlight_label="retired (was store_and_fwd_flag), IDs skip from 6 to 8"
)

### Re-add Column with the Same Name

This is where field IDs shine — old data won't be resurrected.

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.nyc_taxi
    ADD COLUMN store_and_fwd_flag STRING
""")

print("store_and_fwd_flag column re-added!")

In [ ]:
print("Field IDs after re-adding store_and_fwd_flag:")
show_iceberg_schema(
    "polaris.evolution.nyc_taxi",
    highlight_field_id=22,
    highlight_label="new ID for re-added column (not 7!)"
)

print("\nOld store_and_fwd_flag data is NOT resurrected:")
spark.sql("""
    SELECT VendorID, fare_amount, store_and_fwd_flag
    FROM polaris.evolution.nyc_taxi
    LIMIT 5
""").show()

The re-added `store_and_fwd_flag` got a **new** field ID (not 7). Old files still have field 7, but nothing in the current schema maps to it → old data invisible.

**What happened?**
- Original `store_and_fwd_flag` = field ID **7**, retired on drop (gap in ID sequence).
- Re-added column got a fresh field ID.
- Old files still hold the column under ID 7, but the current schema maps the name to the new ID → old data invisible.

### Try It: Test Resurrection Prevention

Drop a column (e.g. `extra`), note its field ID, re-add it by the same name. Confirm via `show_iceberg_schema` the new ID differs. Does old data come back?

In [ ]:
# my_column = "extra"  # pick any column

# Check the field ID before dropping
# show_iceberg_schema("polaris.evolution.nyc_taxi")

# Drop and re-add
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi DROP COLUMN {my_column}")
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi ADD COLUMN {my_column} DOUBLE")

# Check the new field ID. It should be different!
# show_iceberg_schema("polaris.evolution.nyc_taxi")

# Query: old values should be NULL (not resurrected)
# spark.sql(f"SELECT VendorID, {my_column} FROM polaris.evolution.nyc_taxi LIMIT 5").show()

# Clean up: drop again so the notebook continues cleanly
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi DROP COLUMN {my_column}")

## Part 3: Renaming Columns

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.nyc_taxi
    RENAME COLUMN fare_amount TO base_fare
""")

print("Column renamed from 'fare_amount' to 'base_fare'")

In [ ]:
print("Schema after rename (field ID unchanged, only the name mapping changed):")
show_iceberg_schema(
    "polaris.evolution.nyc_taxi",
    highlight_field_id=11,
    highlight_label="was 'fare_amount', same field ID"
)

In [ ]:
print("Query using new name:")
spark.sql("""
    SELECT VendorID, base_fare, tip_amount
    FROM polaris.evolution.nyc_taxi
    WHERE base_fare > 100
    LIMIT 10
""").show()

Field ID unchanged — only the name mapping moved. All existing data is still reachable via the new name.

### Try It: Rename and Verify

Rename another column (e.g. `tip_amount` → `gratuity`). Confirm via `show_iceberg_schema` the field ID is unchanged. Query by the new name, then rename back.

In [ ]:
# old_name = "tip_amount"
# new_name = "gratuity"

# Rename it
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi RENAME COLUMN {old_name} TO {new_name}")
# show_iceberg_schema("polaris.evolution.nyc_taxi")

# Query with the new name
# spark.sql(f"SELECT VendorID, base_fare, {new_name} FROM polaris.evolution.nyc_taxi LIMIT 5").show()

# Rename back to keep the notebook consistent
# spark.sql(f"ALTER TABLE polaris.evolution.nyc_taxi RENAME COLUMN {new_name} TO {old_name}")

## Part 4: Partition Evolution

Change the partition scheme **without rewriting existing data**. Iceberg records which spec each file was written under; old files keep their layout, new files use the new spec, engines reconcile at read time.

### Create an Unpartitioned Table

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.evolution.taxi_evolving")

spark.sql("""
    CREATE TABLE polaris.evolution.taxi_evolving
    USING iceberg
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.evolution.taxi_evolving").collect()[0][0]
print(f"Unpartitioned table created with {count:,} trips")

In [ ]:
print("Files (unpartitioned):")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           spec_id, record_count
    FROM polaris.evolution.taxi_evolving.files
""").show(truncate=False)

print()
show_partition_specs(
    "polaris.evolution.taxi_evolving",
    highlight_spec_id=0,
    highlight_label="initial (unpartitioned)"
)

### Add Day Partitioning

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    ADD PARTITION FIELD days(tpep_pickup_datetime)
""")

print("Day partitioning added! New writes will be partitioned by day.")

### Insert New Data - Now Partitioned

In [ ]:
spark.sql("""
    INSERT INTO polaris.evolution.taxi_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
""")

print("July data inserted (will be day-partitioned)!")

In [ ]:
print("Files after adding day partitioning (mix of spec 0 and spec 1):")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           spec_id, partition, record_count
    FROM polaris.evolution.taxi_evolving.files
    ORDER BY spec_id, partition
""").show(40, truncate=False)

print()
show_partition_specs(
    "polaris.evolution.taxi_evolving",
    highlight_spec_id=1,
    highlight_label="just added: day(pickup)"
)

Old files stay under spec 0 (unpartitioned); new files use spec 1 (day-partitioned). The `specs` table shows the full history.

**Reading the `partition` column:** it's a unified tuple with one slot per partition field that has **ever existed** on the table. It grows as fields are added; fields that don't apply to a file's spec show `NULL`.

1. **Spec 1 (day)** → tuple `{day}`. Spec 0 files: `{NULL}`. Spec 1 files: `{2023-07-01}`.
2. **Spec 2 (month)** → tuple `{day, month}`. Spec 0: `{NULL, NULL}` · Spec 1: `{2023-07-01, NULL}` · Spec 2: `{NULL, 643}` (643 = Aug 2023, months since epoch).

One consistent shape lets engines compare partition values across specs.

### Change to Month Partitioning

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    DROP PARTITION FIELD days(tpep_pickup_datetime)
""")

spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    ADD PARTITION FIELD months(tpep_pickup_datetime)
""")

print("Partition changed from days to months!")

In [ ]:
spark.sql("""
    INSERT INTO polaris.evolution.taxi_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-08.parquet`
""")

print("August data inserted (will be month-partitioned)!")

In [ ]:
print("Files with three partition specs (5 per spec):")
spark.sql("""
    SELECT filename, spec_id, partition, record_count FROM (
        SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
               spec_id, partition, record_count,
               ROW_NUMBER() OVER (PARTITION BY spec_id ORDER BY partition) as rn
        FROM polaris.evolution.taxi_evolving.files
    ) WHERE rn <= 5
    ORDER BY spec_id, partition
""").show(truncate=False)

print()
show_partition_specs(
    "polaris.evolution.taxi_evolving",
    highlight_spec_id=2,
    highlight_label="just added: month(pickup)"
)

### Revert to Day Partitioning (Spec Reuse)

Switching back to day partitioning does **not** create a new spec — Iceberg recognizes spec 1 already means `days(tpep_pickup_datetime)` and reuses it.

In [ ]:
spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    DROP PARTITION FIELD months(tpep_pickup_datetime)
""")

spark.sql("""
    ALTER TABLE polaris.evolution.taxi_evolving
    ADD PARTITION FIELD days(tpep_pickup_datetime)
""")

print("Switched back to day partitioning!")

In [ ]:
spark.sql("""
    INSERT INTO polaris.evolution.taxi_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-09.parquet`
""")

print("September data inserted (day-partitioned again)!")

In [ ]:
print("Files after reverting to day partitioning (5 per spec):")
spark.sql("""
    SELECT filename, spec_id, partition, record_count FROM (
        SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
               spec_id, partition, record_count,
               ROW_NUMBER() OVER (PARTITION BY spec_id ORDER BY partition) as rn
        FROM polaris.evolution.taxi_evolving.files
    ) WHERE rn <= 5
    ORDER BY spec_id, partition
""").show(truncate=False)

print()
show_partition_specs(
    "polaris.evolution.taxi_evolving",
    highlight_spec_id=1,
    highlight_label="reused (no new spec created!)"
)

Still only **three** spec IDs (0, 1, 2). The new day-partitioned files land under **spec 1** — the same one used earlier. Specs are tracked by definition, so reverting reuses the existing spec.

### Try It: Try a Different Transform

Available transforms: `years`, `months`, `days`, `hours`, `bucket(N, col)`, `truncate(N, col)`.

- **`bucket(N, col)`** — hash into N groups; good for high-cardinality columns.
- **`truncate(N, col)`** — first N chars (strings) or round to nearest N (numbers).

Try `hours(tpep_pickup_datetime)` or `bucket(16, PULocationID)` on `taxi_evolving`. Insert data, then inspect `files` + use `show_partition_specs` to track accumulated specs.

In [ ]:
# my_transform = "hours(tpep_pickup_datetime)"  # or bucket(16, PULocationID), etc.

# First, drop the current partition field
# spark.sql("ALTER TABLE polaris.evolution.taxi_evolving DROP PARTITION FIELD days(tpep_pickup_datetime)")

# Add your partition transform
# spark.sql(f"ALTER TABLE polaris.evolution.taxi_evolving ADD PARTITION FIELD {my_transform}")
# show_partition_specs("polaris.evolution.taxi_evolving")

# Insert some data and inspect the files
# spark.sql("""
#     INSERT INTO polaris.evolution.taxi_evolving
#     SELECT * FROM polaris.evolution.taxi_evolving LIMIT 1000
# """)
# spark.sql("""
#     SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
#            spec_id, partition, record_count
#     FROM polaris.evolution.taxi_evolving.files
#     ORDER BY spec_id DESC
#     LIMIT 10
# """).show(truncate=False)

## Cleanup

In [ ]:
# Optional: Drop tables to start fresh
# spark.sql("DROP TABLE IF EXISTS polaris.evolution.nyc_taxi")
# spark.sql("DROP TABLE IF EXISTS polaris.evolution.taxi_evolving")
# print("Tables dropped!")